In [0]:
# Databricks notebook source
# ===================================================================================
#
# JOB B: GENERIC STREAMING INGESTION JOB
#
# AUTHOR: GitHub Copilot & naveenbanappa
# VERSION: 1.0
#
# PURPOSE:
#   A generic, continuous streaming job that reads data from a source location,
#   unpacks and transforms it according to a DDL-driven contract, and writes it
#   to a target Delta table. It is designed to be self-healing and auditable.
#
# WORKFLOW (FOREACHBATCH):
#   1. Reads a micro-batch of data from the source (e.g., ADLS).
#   2. Checks the `schema_transition` table for the latest approved schema hash.
#   3. If the latest hash differs from the job's locked hash, it initiates a
#      graceful shutdown and triggers the backfill process.
#   4. If hashes match, it fetches the detailed contract from `schema_store_columns`.
#   5. It dynamically builds a schema and applies `from_json` to unpack the raw
#      JSON from the source columns (e.g., 'ReqPayload').
#   6. It stamps each row with pipeline metadata (ingest timestamp, schema hash).
#   7. It writes the transformed data to the target Delta table.
#   8. It logs its own run and micro-batch statistics to dedicated audit tables.
#
# "COMMIT-THEN-CORRECT" LOGIC:
#   On its first successful micro-batch run after a schema change, this job will
#   check if a backfill is 'PENDING'. If so, it will automatically trigger the
#   'ewm_backfill_processor' job via the Databricks API to correct historical data.
#
# ===================================================================================

# COMMAND ----------

# MAGIC %md
# MAGIC ### Section 1: Imports & Job Parameters

# COMMAND ----------

import json
import requests
from pyspark.sql import functions as F
from pyspark.sql.types import StructType

# --- Define widgets to accept job parameters
# Basic configuration
dbutils.widgets.text("target_table_name", "", "1. Full name of the target Delta table")
dbutils.widgets.text("source_data_path", "", "2. Path to the raw source data in ADLS")
dbutils.widgets.text("control_table_schema", "mq_gmdf_dp_poc.metadata_ddl", "3. UC Schema for Control & Audit Tables")
dbutils.widgets.text("checkpoint_location_root", "/tmp/checkpoints/", "4. Root path for streaming checkpoints")

# --- NEW: Parameters for secure, generic access to ADLS ---
dbutils.widgets.text("storage_account_name", "", "5. ADLS Storage Account Name")
dbutils.widgets.text("secret_scope_name", "", "6. Databricks Secret Scope Name")
dbutils.widgets.text("secret_key_name", "", "7. Key name within the Secret Scope for ADLS access")


# COMMAND ----------

# MAGIC %md
# MAGIC ### Section 2: ADLS & Databricks API Configuration
# MAGIC This section securely configures access to the ADLS source data and the Databricks API for triggering the backfill job.

# COMMAND ----------

# --- Fetch basic parameters from widgets
target_table_name = dbutils.widgets.get("target_table_name")
source_data_path = dbutils.widgets.get("source_data_path")
control_table_schema = dbutils.widgets.get("control_table_schema")
checkpoint_location_root = dbutils.widgets.get("checkpoint_location_root")

# --- Fetch the NEW secure access parameters from widgets
ADLS_STORAGE_ACCOUNT_NAME = dbutils.widgets.get("storage_account_name")
secret_scope = dbutils.widgets.get("secret_scope_name")
secret_key_name = dbutils.widgets.get("secret_key_name")

# --- Validate that the new secure access parameters have been provided
if not all([ADLS_STORAGE_ACCOUNT_NAME, secret_scope, secret_key_name]):
    raise ValueError("FATAL: The parameters 'storage_account_name', 'secret_scope_name', and 'secret_key_name' are all required for secure data access.")

# --- Define all control/audit table names
schema_transition_table = f"{control_table_schema}.schema_transition"
schema_store_columns_table = f"{control_table_schema}.schema_store_columns"
job_b_run_log_table = f"{control_table_schema}.job_b_run_log"
job_b_microbatch_log_table = f"{control_table_schema}.job_b_microbatch_log"

# === 1. ADLS Storage Authentication using Databricks Secrets ===
# This is now fully generic. The notebook securely retrieves the key
# based on the parameters passed to the job.
spark.conf.set(
    f"fs.azure.account.key.{ADLS_STORAGE_ACCOUNT_NAME}.dfs.core.windows.net",
    dbutils.secrets.get(scope=secret_scope, key=secret_key_name)
)

# === 2. Databricks API Configuration ===
# The information required to call the Databricks Jobs API.
DATABRICKS_HOST = spark.conf.get("spark.databricks.workspaceUrl")
BACKFILL_JOB_NAME = "ewm_backfill_processor" # As you specified

# --- COMMENTED SECTION FOR SERVICE PRINCIPAL ---
# (This part remains the same)
#
# SP_SECRET_SCOPE = "your_sp_scope"
# SP_SECRET_KEY = "your_sp_api_token_key"
# API_TOKEN = dbutils.secrets.get(scope=SP_SECRET_SCOPE, key=SP_SECRET_KEY)
#
# HEADERS = {
#     "Authorization": f"Bearer {API_TOKEN}",
#     "Content-Type": "application/json"
# }

print("✅ Configuration and Authentication settings are defined from job parameters.")

# COMMAND ----------

# MAGIC %md
# MAGIC ### Section 3: The `StreamingProcessor` Class
# MAGIC This class encapsulates all the logic for the `foreachBatch` operation. An instance of this class is created before the stream starts, and its `process` method is called for every micro-batch.

# COMMAND ----------

class StreamingProcessor:
    """
    A class to manage the state and logic of a DDL-driven streaming micro-batch.
    """
    def __init__(self, target_table, locked_hash, prev_hash, backfill_job_name, dbx_host):
        self.target_table = target_table
        self.locked_schema_hash = locked_hash
        self.previous_schema_hash = prev_hash
        self.backfill_job_name = backfill_job_name
        self.databricks_host = dbx_host
        self.is_first_batch = True
        self.job_id_cache = None
        print(f"StreamingProcessor initialized for '{target_table}' with locked hash '{locked_hash}'.")

    def _get_backfill_job_id(self):
        """Finds the job ID for the backfill job by its name. Caches the result."""
        if self.job_id_cache:
            return self.job_id_cache

        print("   - Searching for Backfill Job ID by name...")
        # NOTE: This API call is commented out as it requires a valid token.
        # job_list_url = f"https://{self.databricks_host}/api/2.1/jobs/list"
        # response = requests.get(job_list_url, headers=HEADERS)
        # response.raise_for_status()
        # jobs = response.json().get("jobs", [])
        # for job in jobs:
        #     if job["settings"]["name"] == self.backfill_job_name:
        #         self.job_id_cache = job["job_id"]
        #         print(f"   - Found Job ID {self.job_id_cache} for '{self.backfill_job_name}'.")
        #         return self.job_id_cache
        
        # This is a placeholder since the above is commented out.
        # In a real run, uncommenting the logic above will find the ID.
        print("   - WARNING: API call is commented out. Cannot find Backfill Job ID.")
        return None # In a real run, this would raise an error if not found.

    def _trigger_backfill(self):
        """Triggers the backfill job via the Databricks API."""
        job_id = self._get_backfill_job_id()
        if not job_id:
            print("   - ERROR: Could not trigger backfill because Job ID was not found.")
            return

        print(f"   - Triggering backfill for job ID {job_id}...")
        run_now_url = f"https://{self.databricks_host}/api/2.1/jobs/run-now"
        
        payload = {
            "job_id": job_id,
            "notebook_params": {
                "target_table_name": self.target_table,
                "old_schema_hash": self.previous_schema_hash,
                "new_schema_hash": self.locked_schema_hash,
                "run_mode": "WET_RUN",
                "control_table_schema": control_table_schema
            }
        }
        
        # NOTE: This API call is commented out. When the SP is configured, this can be enabled.
        # try:
        #     response = requests.post(run_now_url, headers=HEADERS, data=json.dumps(payload))
        #     response.raise_for_status()
        #     print(f"   - Successfully triggered backfill job run. Response: {response.json()}")
        #     spark.sql(f"UPDATE {schema_transition_table} SET backfill_status = 'IN_PROGRESS' WHERE target_table_name = '{self.target_table}' AND schema_hash = '{self.locked_schema_hash}'")
        # except Exception as e:
        #     print(f"   - FATAL: Failed to trigger backfill job. Error: {e}")
        #     # You might want to add alerting here.
        
        print("   - NOTE: Backfill trigger call is currently COMMENTED OUT.")


    def _check_and_trigger_backfill(self):
        """Checks if a backfill is pending and triggers it if so."""
        print("   - First batch complete. Checking for pending backfill...")
        status_row = spark.sql(f"SELECT backfill_status FROM {schema_transition_table} WHERE target_table_name = '{self.target_table}' AND schema_hash = '{self.locked_schema_hash}'").first()
        
        if status_row and status_row["backfill_status"] == "PENDING":
            print(f"   - Backfill status is 'PENDING' for hash {self.locked_schema_hash}. Triggering backfill job.")
            self._trigger_backfill()
        else:
            print("   - Backfill not pending or already complete. No action needed.")

    def process(self, micro_batch_df, batch_id):
        """The main processing function for each micro-batch."""
        print(f"\n--- Processing Micro-Batch ID: {batch_id} ---")
        
        # 1. Check for schema change signal
        latest_hash_row = spark.sql(f"SELECT schema_hash FROM {schema_transition_table} WHERE target_table_name = '{self.target_table}' ORDER BY event_ts DESC LIMIT 1").first()
        latest_approved_hash = latest_hash_row["schema_hash"] if latest_hash_row else ""
        
        if latest_approved_hash != self.locked_schema_hash:
            raise Exception(f"Schema change detected! From {self.locked_schema_hash} to {latest_approved_hash}. Shutting down gracefully.")

        # 2. On first successful batch, check if we need to trigger the backfill
        if self.is_first_batch:
            self._check_and_trigger_backfill()
            self.is_first_batch = False

        # 3. Fetch the contract and build the unpacking logic
        print("   - Fetching contract from schema store...")
        contract_cols = spark.table(schema_store_columns_table).filter(
            (F.col("target_table_name") == self.target_table) & (F.col("contract_hash") == self.locked_schema_hash)
        ).collect()
        
        final_df = micro_batch_df
        
        # Group by source column (e.g., 'ReqPayload', 'RespPayload') to apply from_json once per source
        source_json_cols = {}
        for row in contract_cols:
            if row['source_column'] not in source_json_cols:
                source_json_cols[row['source_column']] = []
            source_json_cols[row['source_column']].append(row)
            
        for source_col, fields in source_json_cols.items():
            schema = StructType()
            for field in fields:
                schema.add(field['field_name_in_struct'], field['field_type'], True)
            
            unpacked_col_name = f"unpacked_{source_col}"
            final_df = final_df.withColumn(unpacked_col_name, F.from_json(F.col(source_col), schema))
            
            # Create the final columns
            for field in fields:
                final_df = final_df.withColumn(field['field_name_in_struct'], F.col(f"{unpacked_col_name}.{field['field_name_in_struct']}"))
            
            final_df = final_df.drop(unpacked_col_name)

        # 4. Add pipeline metadata
        final_df = final_df.withColumn("ingest_ts", F.current_timestamp())
        final_df = final_df.withColumn("schema_hash_applied", F.lit(self.locked_schema_hash))

        # 5. Select only the columns defined in the target table to avoid schema mismatch errors
        target_table_cols = [field.name for field in spark.table(self.target_table).schema.fields]
        final_df = final_df.select(*target_table_cols)
        
        # 6. Write to Delta table
        print(f"   - Writing {final_df.count()} rows to '{self.target_table}'...")
        final_df.write.format("delta").mode("append").saveAsTable(self.target_table)
        
        # NOTE: Logic for writing to `job_b_microbatch_log` would go here.
        print(f"--- Batch {batch_id} complete. ---")

# COMMAND ----------

# MAGIC %md
# MAGIC ### Section 4: Stream Initialization & Execution

# COMMAND ----------

# --- 1. Get the latest and previous schema hashes to initialize the processor
print("🚀 Initializing Streaming Job...")
hashes_df = spark.sql(f"""
    SELECT schema_hash, event_ts
    FROM {schema_transition_table}
    WHERE target_table_name = '{target_table_name}'
    ORDER BY event_ts DESC
    LIMIT 2
""")
hashes = hashes_df.collect()

if not hashes:
    raise Exception(f"FATAL: No schema has been registered for target table '{target_table_name}' in '{schema_transition_table}'. Run Job A first.")

LOCKED_SCHEMA_HASH = hashes[0]["schema_hash"]
PREVIOUS_SCHEMA_HASH = hashes[1]["schema_hash"] if len(hashes) > 1 else ""

print(f"   - Locking stream to schema hash: {LOCKED_SCHEMA_HASH}")
print(f"   - Previous schema hash was: {PREVIOUS_SCHEMA_HASH}")

# --- 2. Instantiate the processor class
processor = StreamingProcessor(
    target_table=target_table_name,
    locked_hash=LOCKED_SCHEMA_HASH,
    prev_hash=PREVIOUS_SCHEMA_HASH,
    backfill_job_name=BACKFILL_JOB_NAME,
    dbx_host=DATABRICKS_HOST
)

# --- 3. Define the source stream reader
# This example uses Auto Loader (cloudFiles) to ingest JSON files efficiently.
inputStream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", f"{checkpoint_location_root}{target_table_name.replace('.', '_')}/schema")
    .load(source_data_path)
)

# --- 4. Define the sink stream writer
# The `.foreachBatch` function is where all our custom logic is applied.
# The checkpoint location is crucial for stream recovery and must be unique per stream.
checkpoint_path = f"{checkpoint_location_root}{target_table_name.replace('.', '_')}/data"

query = (
    inputStream.writeStream
    .foreachBatch(processor.process)
    .option("checkpointLocation", checkpoint_path)
    .trigger(processingTime='30 seconds') # CORRECTED: This sets the micro-batch interval.
    .start()
)

print(f"✅ Streaming query '{query.name}' started successfully with a 30-second processing time trigger.")
print(f"   - Checkpoint Location: {checkpoint_path}")

# query.awaitTermination() # This line is needed if running interactively to keep the stream alive.
# In a job, the notebook will run indefinitely until the stream is stopped or fails.

